
> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.

# Future-Compatible Historical Retrieval
## Final Confirmatory Benchmark Experiment

This notebook implements the **final confirmatory benchmark experiment**. No new architecture or post-hoc method is introduced here.

---

# Scientific question

The experiment is not merely asking whether the learned retriever beats Pattern retrieval. The central question is:

\[
\boxed{\text{What makes a historical example predictively relevant?}}
\]

The working hypothesis is that historical relevance contains distinct components:

\[
\boxed{\text{surface similarity}\neq\text{candidate-level utility}\neq\text{query-specific compatibility}}
\]

---

# Final datasets

We evaluate:

- Electricity
- Traffic
- Exchange
- Solar

Exchange and Solar serve as additional confirmatory datasets. Electricity and Traffic are re-evaluated under the frozen **same-channel + fixed train-scale + strong shuffled-control** protocol.

---

# Frozen protocol

\[
L=96,\qquad H\in\{24,48,96\},\qquad M=100,\qquad K=10.
\]

The future target is

\[
\boxed{\mathbf y_t=\mathbf x^{\mathrm{train\mbox{-}norm}}_{t+1:t+H}-x^{\mathrm{train\mbox{-}norm}}_t}
\]

and is not divided again by the local past standard deviation.

Candidate policy:

\[
\boxed{\text{Same-channel retrieval}}
\]

Methods:

1. Pattern
2. Candidate Prior
3. Future-Compatible Learned
4. Within-channel Shuffled-Future

---

# Candidate Prior

For candidate \(i\), define its average historical future distance within the corresponding channel as

\[
b_i=\mathbb E_{q'\sim\mathcal Q_{\mathrm{historical},c(i)}}\left[\frac{1}{H}\|\mathbf y_{q'}-\mathbf y_i\|_2^2\right].
\]

The Candidate Prior score is

\[
s_{\mathrm{prior}}(i)=-b_i.
\]

Candidate futures are already observed in historical memory, so this offline prior does not use the current query future.

---

# Query-specific supervision

During training, the learned retriever uses

\[
d(q,i)=\frac{1}{H}\|\mathbf y_q-\mathbf y_i\|_2^2
\]

as relevance supervision. Query futures and candidate futures are never input to the inference-time scoring function.

---

# Strong shuffled control

The Shuffled model uses the same architecture, candidate pool, observable context, and optimization. The only difference is that training query futures are shifted by a non-zero cyclic permutation **within each channel**. This preserves the channel-wise marginal distribution and fixed scaling while breaking the correct query--future correspondence.

---

# Interpretation

Success does not require Learned to be best on every dataset. The controls diagnose different relevance regimes:

### Query-specific relevance
\[
\text{Learned}<\text{Shuffled}
\]

### Candidate-global utility
\[
\text{CandidatePrior}<\text{Pattern}
\]

### Learned relevance beyond the global prior
\[
\text{Learned}<\text{CandidatePrior}
\]

The benchmark recipe is frozen after this experiment irrespective of the observed regime.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


## 0. Imports and frozen configuration

In [ ]:

from pathlib import Path

import gzip
import hashlib
import math
import random
import shutil
import urllib.request
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATA_ROOT = REPO_DATA_ROOT

DATA_PATHS = {
    "Electricity":
        DATA_ROOT /
        "electricity/electricity.csv",

    "Traffic":
        DATA_ROOT /
        "traffic/traffic.csv",

    "Exchange":
        DATA_ROOT /
        "exchange_rate/exchange_rate.csv",

    "Solar":
        DATA_ROOT /
        "Solar/solar_AL.txt",
}

RESULT_DIR = REPO_WORK_ROOT / "final_confirmatory"

CACHE_DIR = (
    RESULT_DIR /
    "cache"
)

MODEL_DIR = (
    RESULT_DIR /
    "models"
)

for p in [
    DATA_ROOT,
    RESULT_DIR,
    CACHE_DIR,
    MODEL_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

DATASET_NAMES = [
    "Electricity",
    "Traffic",
    "Exchange",
    "Solar",
]

HORIZONS = [
    24,
    48,
    96,
]

SEQ_LEN = 96

TOP_M = 100
TOP_K = 10

TAU_Y = 0.50

# Final confirmatory replication.
SEEDS = [
    0,
    1,
    2,
    3,
    4,
]

# Same settings across all four datasets.
MAX_MEMORY_WINDOWS = 50000
MAX_TRAIN_QUERIES = 10000
MAX_VAL_QUERIES = 10000
MAX_TEST_QUERIES = 12000

TRAIN_BATCH = 128
EVAL_BATCH = 256
SEARCH_QUERY_BATCH = 512

MAX_EPOCHS = 30
PATIENCE = 6

LR = 1e-3
WEIGHT_DECAY = 1e-4

BLOCK_ANCHORS = 10
N_BOOT = 5000

EPS = 1e-8

# Large homogeneous datasets use a deterministic 32-channel subset
# to keep candidate construction comparable to the previous study.
# Solar is kept in full because it has 137 channels.
MAX_CHANNELS = {
    "Electricity": 32,
    "Traffic": 32,
    "Exchange": None,
    "Solar": None,
}

# The small 8-variate Exchange dataset benefits from denser anchors.
# This is frozen before looking at results.
WINDOW_STRIDE = {
    "Electricity": 24,
    "Traffic": 24,
    "Exchange": 8,
    "Solar": 24,
}

DATASET_SEED = {
    "Electricity": 3303,
    "Traffic": 4404,
    "Exchange": 5505,
    "Solar": 6606,
}

FORCE_DOWNLOAD = False
FORCE_REBUILD_WINDOWS = False
FORCE_REBUILD_TOPM = False
FORCE_RETRAIN = False

print(
    "Device:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        )
    )

    print(
        "GPU memory (GB):",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory /
            1024**3,
            1,
        )
    )

print(
    "Result directory:",
    RESULT_DIR
)



## 1. Dataset download

### Electricity / Traffic / Exchange

These files are downloaded from the THUML Time-Series-Library dataset repository
when they are not already available locally.

### Solar

The standard Solar-Energy benchmark file is `solar_AL.txt`.

Primary source used here:

```text
laiguokun/multivariate-time-series-data
solar-energy/solar_AL.txt.gz
```

The gzip MD5 published for this standard file is checked before extraction.

A Hugging Face mirror is included only as a fallback.

No dataset is silently overwritten unless `FORCE_DOWNLOAD=True`.


In [ ]:

TSLIB_BASE = (
    "https://huggingface.co/datasets/"
    "thuml/Time-Series-Library/resolve/main"
)

DOWNLOAD_URLS = {
    "Electricity":
        (
            TSLIB_BASE +
            "/electricity/electricity.csv"
        ),

    "Traffic":
        (
            TSLIB_BASE +
            "/traffic/traffic.csv"
        ),

    "Exchange":
        (
            TSLIB_BASE +
            "/exchange_rate/exchange_rate.csv"
        ),
}

SOLAR_GZ_URL = (
    "https://raw.githubusercontent.com/"
    "laiguokun/multivariate-time-series-data/"
    "master/solar-energy/solar_AL.txt.gz"
)

SOLAR_GZ_MD5 = (
    "41ef7fdc958c2ca3fac9cd06d6227073"
)

SOLAR_FALLBACK_URL = (
    "https://huggingface.co/datasets/"
    "pkr7098/time-series-forecasting-datasets/"
    "resolve/main/solar_AL.txt"
)

SOLAR_FALLBACK_SHA256 = (
    "230327ef72d2abb387939d4a35d6fd34"
    "f1066071bc7c40ce7ecf5531a0122ac2"
)


In [ ]:

def file_digest(
    path,
    algorithm,
    chunk_size=1024 * 1024,
):
    h = hashlib.new(
        algorithm
    )

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def download_file(
    url,
    output_path,
):
    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = output_path.with_suffix(
        output_path.suffix +
        ".part"
    )

    if tmp.exists():
        tmp.unlink()

    print(
        "Downloading:",
        url
    )

    urllib.request.urlretrieve(
        url,
        tmp,
    )

    tmp.replace(
        output_path
    )

    print(
        "Saved:",
        output_path,
        "| MB:",
        round(
            output_path.stat().st_size /
            1024**2,
            2,
        )
    )


def ensure_tslib_csv(
    dataset_name,
):
    path = DATA_PATHS[
        dataset_name
    ]

    if (
        path.exists()
        and
        not FORCE_DOWNLOAD
    ):

        print(
            dataset_name,
            "| existing:",
            path
        )

        return

    download_file(
        DOWNLOAD_URLS[
            dataset_name
        ],
        path,
    )


for name in [
    "Electricity",
    "Traffic",
    "Exchange",
]:
    ensure_tslib_csv(
        name
    )


In [ ]:

def ensure_solar():
    final_path = DATA_PATHS[
        "Solar"
    ]

    if (
        final_path.exists()
        and
        not FORCE_DOWNLOAD
    ):

        print(
            "Solar | existing:",
            final_path
        )

        return

    final_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    gz_path = (
        final_path.parent /
        "solar_AL.txt.gz"
    )

    try:

        download_file(
            SOLAR_GZ_URL,
            gz_path,
        )

        observed_md5 = file_digest(
            gz_path,
            "md5",
        )

        assert (
            observed_md5 ==
            SOLAR_GZ_MD5
        ), (
            "Solar gzip MD5 mismatch",
            observed_md5,
        )

        with gzip.open(
            gz_path,
            "rb",
        ) as src:

            with open(
                final_path,
                "wb",
            ) as dst:

                shutil.copyfileobj(
                    src,
                    dst,
                )

        print(
            "Solar extracted:",
            final_path
        )

    except Exception as e:

        print(
            "Primary Solar download failed:",
            repr(
                e
            )
        )

        print(
            "Trying fallback mirror."
        )

        if gz_path.exists():
            gz_path.unlink()

        download_file(
            SOLAR_FALLBACK_URL,
            final_path,
        )

        observed_sha256 = file_digest(
            final_path,
            "sha256",
        )

        assert (
            observed_sha256 ==
            SOLAR_FALLBACK_SHA256
        ), (
            "Solar fallback SHA256 mismatch",
            observed_sha256,
        )

    assert final_path.exists()


ensure_solar()


## 2. Load and validate all datasets

In [ ]:

def load_standard_csv(
    path,
):
    df = pd.read_csv(
        path
    )

    timestamp_cols = [
        c
        for c in df.columns
        if str(
            c
        ).lower() in {
            "date",
            "datetime",
            "timestamp",
            "time",
        }
    ]

    x = (
        df
        .drop(
            columns=timestamp_cols,
            errors="ignore",
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    good_cols = [
        c
        for c in x.columns
        if x[
            c
        ].notna().mean() >
        0.99
    ]

    x = x[
        good_cols
    ]

    x = (
        x
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    assert x.shape[
        1
    ] > 0

    assert np.isfinite(
        x.to_numpy(
            dtype=np.float32
        )
    ).all()

    return x


def load_solar_txt(
    path,
):
    # Standard solar_AL.txt is comma separated and headerless.
    x = pd.read_csv(
        path,
        header=None,
    )

    # Defensive fallback for whitespace-delimited mirrors.
    if x.shape[
        1
    ] == 1:

        x = pd.read_csv(
            path,
            header=None,
            sep=r"\s+",
        )

    x = x.apply(
        pd.to_numeric,
        errors="coerce",
    )

    x = (
        x
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    assert x.shape[
        1
    ] == 137, (
        "Expected 137 Solar channels, got",
        x.shape[
            1
        ],
    )

    x.columns = [
        f"Solar_{i:03d}"
        for i in range(
            x.shape[
                1
            ]
        )
    ]

    assert np.isfinite(
        x.to_numpy(
            dtype=np.float32
        )
    ).all()

    return x


RAW = {
    "Electricity":
        load_standard_csv(
            DATA_PATHS[
                "Electricity"
            ]
        ),

    "Traffic":
        load_standard_csv(
            DATA_PATHS[
                "Traffic"
            ]
        ),

    "Exchange":
        load_standard_csv(
            DATA_PATHS[
                "Exchange"
            ]
        ),

    "Solar":
        load_solar_txt(
            DATA_PATHS[
                "Solar"
            ]
        ),
}

for name, df in RAW.items():

    print(
        f"{name:12s}",
        "| rows:",
        len(
            df
        ),
        "| channels:",
        df.shape[
            1
        ],
    )


## 3. Data provenance manifest

In [ ]:

manifest_rows = []

for name in DATASET_NAMES:

    path = DATA_PATHS[
        name
    ]

    manifest_rows.append({
        "Dataset":
            name,

        "Path":
            str(
                path
            ),

        "Rows":
            len(
                RAW[
                    name
                ]
            ),

        "NumericChannels":
            RAW[
                name
            ].shape[
                1
            ],

        "FileSizeMB":
            round(
                path.stat().st_size /
                1024**2,
                4,
            ),

        "SHA256":
            file_digest(
                path,
                "sha256",
            ),
    })

manifest_table = pd.DataFrame(
    manifest_rows
)

display(
    manifest_table
)

manifest_table.to_csv(
    RESULT_DIR /
    "00_data_manifest.csv",
    index=False,
)


## 4. Frozen temporal split

In [ ]:

def split_boundaries(
    n,
):
    train_end = int(
        0.70 *
        n
    )

    val_end = int(
        0.80 *
        n
    )

    return {
        "train_end":
            train_end,

        "val_end":
            val_end,

        "test_end":
            n,

        # Retrieval memory/query split within the training region.
        "inner_memory_end":
            int(
                0.60 *
                train_end
            ),
    }


SPLITS = {
    name:
        split_boundaries(
            len(
                RAW[
                    name
                ]
            )
        )
    for name in DATASET_NAMES
}


## 5. Remove degenerate channels and select frozen channel subset

In [ ]:

SELECTED_CHANNELS = {}
CHANNEL_QUALITY_ROWS = []

for name in DATASET_NAMES:

    df = RAW[
        name
    ]

    train_end = SPLITS[
        name
    ][
        "train_end"
    ]

    train = df.iloc[
        :train_end
    ]

    train_std = train.std(
        axis=0,
        ddof=0,
    )

    valid_cols = [
        c
        for c in df.columns
        if (
            np.isfinite(
                train_std[
                    c
                ]
            )
            and
            train_std[
                c
            ] >
            1e-6
        )
    ]

    max_c = MAX_CHANNELS[
        name
    ]

    if (
        max_c is not None
        and
        len(
            valid_cols
        ) >
        max_c
    ):

        idx = np.linspace(
            0,
            len(
                valid_cols
            ) -
            1,
            max_c,
            dtype=int,
        )

        selected = [
            valid_cols[
                i
            ]
            for i in idx
        ]

    else:

        selected = valid_cols

    SELECTED_CHANNELS[
        name
    ] = selected

    CHANNEL_QUALITY_ROWS.append({
        "Dataset":
            name,

        "OriginalChannels":
            df.shape[
                1
            ],

        "NondegenerateTrainChannels":
            len(
                valid_cols
            ),

        "UsedChannels":
            len(
                selected
            ),

        "MaxChannelsRule":
            MAX_CHANNELS[
                name
            ],
    })

channel_quality_table = pd.DataFrame(
    CHANNEL_QUALITY_ROWS
)

display(
    channel_quality_table
)

channel_quality_table.to_csv(
    RESULT_DIR /
    "01_channel_selection.csv",
    index=False,
)


## 6. Train-only channel normalization

In [ ]:

CHANNEL_NORMALIZED = {}
CHANNEL_STATS = {}

for name in DATASET_NAMES:

    cols = SELECTED_CHANNELS[
        name
    ]

    df = RAW[
        name
    ][
        cols
    ]

    train_end = SPLITS[
        name
    ][
        "train_end"
    ]

    train = df.iloc[
        :train_end
    ]

    mean = train.mean(
        axis=0
    ).to_numpy(
        dtype=np.float32
    )

    std = train.std(
        axis=0,
        ddof=0,
    ).to_numpy(
        dtype=np.float32
    )

    assert np.all(
        std >
        1e-6
    )

    arr = df.to_numpy(
        dtype=np.float32
    )

    z = (
        arr -
        mean[
            None,
            :
        ]
    ) / std[
        None,
        :
    ]

    assert np.isfinite(
        z
    ).all()

    CHANNEL_NORMALIZED[
        name
    ] = z.astype(
        np.float32
    )

    CHANNEL_STATS[
        name
    ] = {
        "mean":
            mean,

        "std":
            std,
    }


## 7. Observable context features

In [ ]:

def safe_autocorr_lag1(
    x,
):
    a = (
        x[
            :-1
        ]
        -
        x[
            :-1
        ].mean()
    )

    b = (
        x[
            1:
        ]
        -
        x[
            1:
        ].mean()
    )

    den = (
        np.sqrt(
            np.sum(
                a ** 2
            )
            *
            np.sum(
                b ** 2
            )
        )
        +
        EPS
    )

    return float(
        np.sum(
            a *
            b
        ) /
        den
    )


def normalized_slope(
    x,
):
    n = len(
        x
    )

    t = np.linspace(
        -1.0,
        1.0,
        n,
        dtype=np.float32,
    )

    t = (
        t -
        t.mean()
    )

    xc = (
        x -
        x.mean()
    )

    slope = (
        np.sum(
            t *
            xc
        )
        /
        (
            np.sum(
                t ** 2
            )
            +
            EPS
        )
    )

    return float(
        slope
        /
        (
            x.std()
            +
            EPS
        )
    )


def generic_context(
    past,
):
    past = np.asarray(
        past,
        dtype=np.float32,
    )

    L = len(
        past
    )

    short = max(
        8,
        L //
        4
    )

    std_full = (
        float(
            np.std(
                past
            )
        )
        +
        EPS
    )

    current_level = (
        past[
            -1
        ]
        -
        np.mean(
            past
        )
    ) / std_full

    mean_gap = (
        np.mean(
            past[
                -short:
            ]
        )
        -
        np.mean(
            past
        )
    ) / std_full

    short_change = (
        past[
            -1
        ]
        -
        past[
            -short
        ]
    ) / std_full

    long_change = (
        past[
            -1
        ]
        -
        past[
            0
        ]
    ) / std_full

    d_full = np.diff(
        past
    )

    d_short = np.diff(
        past[
            -short:
        ]
    )

    vol_ratio = (
        np.std(
            d_short
        )
        +
        EPS
    ) / (
        np.std(
            d_full
        )
        +
        EPS
    )

    slope = normalized_slope(
        past
    )

    ac1 = safe_autocorr_lag1(
        past
    )

    return np.asarray(
        [
            current_level,
            mean_gap,
            short_change,
            long_change,
            vol_ratio,
            slope,
            ac1,
        ],
        dtype=np.float32,
    )


CONTEXT_DIM = 7


## 8. Pattern vector and fixed train-scale future target

In [ ]:

def pattern_vector(
    past,
):
    x = np.asarray(
        past,
        dtype=np.float32,
    )

    x = (
        x -
        x.mean()
    )

    norm = np.linalg.norm(
        x
    )

    if norm < EPS:

        return np.zeros_like(
            x
        )

    return (
        x /
        norm
    ).astype(
        np.float32
    )


def build_windows(
    dataset_name,
    H,
):
    arr = CHANNEL_NORMALIZED[
        dataset_name
    ]

    channels = SELECTED_CHANNELS[
        dataset_name
    ]

    stride = WINDOW_STRIDE[
        dataset_name
    ]

    n_time, n_chan = (
        arr.shape
    )

    meta_rows = []
    pattern_rows = []
    context_rows = []
    future_rows = []

    skipped_low_variance = 0

    for cidx in range(
        n_chan
    ):

        series = arr[
            :,
            cidx
        ]

        for anchor in range(
            SEQ_LEN,
            n_time -
            H +
            1,
            stride,
        ):

            past = series[
                anchor -
                SEQ_LEN:
                anchor
            ]

            # Keep the same pre-specified low-variance exclusion
            # used in the previous generic benchmark protocol.
            if float(
                np.std(
                    past
                )
            ) < 1e-5:

                skipped_low_variance += 1
                continue

            future_raw = series[
                anchor:
                anchor +
                H
            ]

            # Fixed train-scale target:
            # channel has already been normalized using train-only mean/std.
            future = (
                future_raw
                -
                past[
                    -1
                ]
            ).astype(
                np.float32
            )

            meta_rows.append(
                (
                    cidx,
                    channels[
                        cidx
                    ],
                    anchor,
                    anchor +
                    H -
                    1,
                )
            )

            pattern_rows.append(
                pattern_vector(
                    past
                )
            )

            context_rows.append(
                generic_context(
                    past
                )
            )

            future_rows.append(
                future
            )

    assert len(
        meta_rows
    ) > 0

    return {
        "meta":
            pd.DataFrame(
                meta_rows,
                columns=[
                    "ChannelIndex",
                    "Channel",
                    "Anchor",
                    "FutureEnd",
                ],
            ),

        "pattern":
            np.stack(
                pattern_rows
            ).astype(
                np.float32
            ),

        "context":
            np.stack(
                context_rows
            ).astype(
                np.float32
            ),

        "future":
            np.stack(
                future_rows
            ).astype(
                np.float32
            ),

        "skipped_low_variance":
            skipped_low_variance,
    }


## 9. Build/load window caches

In [ ]:

WINDOWS = {}
window_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        npz_path = (
            CACHE_DIR /
            f"{dataset_name}_H{H}_windows.npz"
        )

        meta_path = (
            CACHE_DIR /
            f"{dataset_name}_H{H}_meta.csv.gz"
        )

        info_path = (
            CACHE_DIR /
            f"{dataset_name}_H{H}_window_info.csv"
        )

        if (
            npz_path.exists()
            and
            meta_path.exists()
            and
            not FORCE_REBUILD_WINDOWS
        ):

            z = np.load(
                npz_path
            )

            meta = pd.read_csv(
                meta_path
            )

            w = {
                "meta":
                    meta,

                "pattern":
                    z[
                        "pattern"
                    ].astype(
                        np.float32
                    ),

                "context":
                    z[
                        "context"
                    ].astype(
                        np.float32
                    ),

                "future":
                    z[
                        "future"
                    ].astype(
                        np.float32
                    ),

                "skipped_low_variance":
                    (
                        int(
                            pd.read_csv(
                                info_path
                            )[
                                "SkippedLowVariance"
                            ].iloc[
                                0
                            ]
                        )
                        if info_path.exists()
                        else
                        -1
                    ),
            }

            source = "loaded"

        else:

            print(
                "Building windows:",
                dataset_name,
                "H=",
                H
            )

            w = build_windows(
                dataset_name,
                H,
            )

            np.savez_compressed(
                npz_path,
                pattern=w[
                    "pattern"
                ],
                context=w[
                    "context"
                ],
                future=w[
                    "future"
                ],
            )

            w[
                "meta"
            ].to_csv(
                meta_path,
                index=False,
                compression="gzip",
            )

            pd.DataFrame(
                [
                    {
                        "Dataset":
                            dataset_name,

                        "Horizon":
                            H,

                        "SkippedLowVariance":
                            w[
                                "skipped_low_variance"
                            ],
                    }
                ]
            ).to_csv(
                info_path,
                index=False,
            )

            source = "built"

        WINDOWS[
            key
        ] = w

        window_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "Stride":
                WINDOW_STRIDE[
                    dataset_name
                ],

            "Windows":
                len(
                    w[
                        "meta"
                    ]
                ),

            "SkippedLowVariance":
                w[
                    "skipped_low_variance"
                ],

            "Source":
                source,
        })

window_table = pd.DataFrame(
    window_rows
)

display(
    window_table
)

window_table.to_csv(
    RESULT_DIR /
    "02_window_summary.csv",
    index=False,
)


## 10. Phase boundaries

In [ ]:

def build_phase_indices(
    dataset_name,
    H,
):
    key = (
        dataset_name,
        H
    )

    meta = WINDOWS[
        key
    ][
        "meta"
    ]

    s = SPLITS[
        dataset_name
    ]

    anchor = meta[
        "Anchor"
    ].to_numpy(
        dtype=np.int64
    )

    future_end = meta[
        "FutureEnd"
    ].to_numpy(
        dtype=np.int64
    )

    return {
        "train_memory":
            np.where(
                future_end
                <
                s[
                    "inner_memory_end"
                ]
            )[0],

        "train_query":
            np.where(
                (
                    anchor
                    >=
                    s[
                        "inner_memory_end"
                    ]
                )
                &
                (
                    future_end
                    <
                    s[
                        "train_end"
                    ]
                )
            )[0],

        "val_memory":
            np.where(
                future_end
                <
                s[
                    "train_end"
                ]
            )[0],

        "val_query":
            np.where(
                (
                    anchor
                    >=
                    s[
                        "train_end"
                    ]
                )
                &
                (
                    future_end
                    <
                    s[
                        "val_end"
                    ]
                )
            )[0],

        "test_memory":
            np.where(
                future_end
                <
                s[
                    "val_end"
                ]
            )[0],

        "test_query":
            np.where(
                anchor
                >=
                s[
                    "val_end"
                ]
            )[0],
    }


FULL_PHASES = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        FULL_PHASES[
            (
                dataset_name,
                H
            )
        ] = build_phase_indices(
            dataset_name,
            H,
        )


## 11. Channel-balanced deterministic sampling

In [ ]:

def balanced_subset(
    meta,
    indices,
    max_n,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(
        indices
    ) <= max_n:

        return np.sort(
            indices
        )

    rng = np.random.default_rng(
        seed
    )

    channels = meta.iloc[
        indices
    ][
        "ChannelIndex"
    ].to_numpy(
        dtype=np.int64
    )

    unique_channels = np.unique(
        channels
    )

    random_channel_order = rng.permutation(
        unique_channels
    )

    base = (
        max_n //
        len(
            unique_channels
        )
    )

    extra = (
        max_n %
        len(
            unique_channels
        )
    )

    chosen_parts = []

    for rank, c in enumerate(
        random_channel_order
    ):

        pos = indices[
            channels ==
            c
        ]

        quota = (
            base
            +
            (
                1
                if rank <
                extra
                else 0
            )
        )

        take = min(
            quota,
            len(
                pos
            ),
        )

        if take > 0:

            chosen_parts.append(
                rng.choice(
                    pos,
                    size=take,
                    replace=False,
                )
            )

    chosen = np.unique(
        np.concatenate(
            chosen_parts
        )
    )

    # Fill any unused quota globally if some channels were too small.
    if len(
        chosen
    ) < max_n:

        remaining = np.setdiff1d(
            indices,
            chosen,
            assume_unique=False,
        )

        add_n = min(
            max_n -
            len(
                chosen
            ),
            len(
                remaining
            ),
        )

        if add_n > 0:

            chosen = np.concatenate(
                [
                    chosen,
                    rng.choice(
                        remaining,
                        size=add_n,
                        replace=False,
                    ),
                ]
            )

    return np.sort(
        chosen.astype(
            np.int64
        )
    )


In [ ]:

TASK_INDICES = {}
sample_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        full = FULL_PHASES[
            key
        ]

        meta = WINDOWS[
            key
        ][
            "meta"
        ]

        base = (
            DATASET_SEED[
                dataset_name
            ]
            +
            H *
            10
        )

        sampled = {
            "train_memory":
                balanced_subset(
                    meta,
                    full[
                        "train_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    1,
                ),

            "train_query":
                balanced_subset(
                    meta,
                    full[
                        "train_query"
                    ],
                    MAX_TRAIN_QUERIES,
                    base +
                    2,
                ),

            "val_memory":
                balanced_subset(
                    meta,
                    full[
                        "val_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    3,
                ),

            "val_query":
                balanced_subset(
                    meta,
                    full[
                        "val_query"
                    ],
                    MAX_VAL_QUERIES,
                    base +
                    4,
                ),

            "test_memory":
                balanced_subset(
                    meta,
                    full[
                        "test_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    5,
                ),

            "test_query":
                balanced_subset(
                    meta,
                    full[
                        "test_query"
                    ],
                    MAX_TEST_QUERIES,
                    base +
                    6,
                ),
        }

        TASK_INDICES[
            key
        ] = sampled

        sample_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            **{
                k:
                    len(
                        v
                    )
                for k, v in sampled.items()
            },
        })

sample_table = pd.DataFrame(
    sample_rows
)

display(
    sample_table
)

sample_table.to_csv(
    RESULT_DIR /
    "03_task_sample_summary.csv",
    index=False,
)


## 12. Same-channel memory coverage check

In [ ]:

coverage_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        meta = WINDOWS[
            key
        ][
            "meta"
        ]

        for phase in [
            "train",
            "val",
            "test",
        ]:

            mem_idx = TASK_INDICES[
                key
            ][
                f"{phase}_memory"
            ]

            q_idx = TASK_INDICES[
                key
            ][
                f"{phase}_query"
            ]

            mem_counts = (
                meta.iloc[
                    mem_idx
                ]
                .groupby(
                    "ChannelIndex"
                )
                .size()
            )

            q_channels = np.unique(
                meta.iloc[
                    q_idx
                ][
                    "ChannelIndex"
                ].to_numpy()
            )

            missing = [
                int(
                    c
                )
                for c in q_channels
                if (
                    c not in mem_counts.index
                    or
                    mem_counts.loc[
                        c
                    ] < TOP_M
                )
            ]

            min_count = min(
                int(
                    mem_counts.loc[
                        c
                    ]
                )
                for c in q_channels
                if c in mem_counts.index
            )

            coverage_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Phase":
                    phase,

                "QueryChannels":
                    len(
                        q_channels
                    ),

                "MinMemoryCandidatesPerQueryChannel":
                    min_count,

                "ChannelsBelowTopM":
                    len(
                        missing
                    ),
            })

            assert len(
                missing
            ) == 0, (
                dataset_name,
                H,
                phase,
                missing[
                    :10
                ],
            )

coverage_table = pd.DataFrame(
    coverage_rows
)

display(
    coverage_table
)

coverage_table.to_csv(
    RESULT_DIR /
    "04_same_channel_coverage.csv",
    index=False,
)


## 13. Context scaling fit on sampled train memory only

In [ ]:

def fit_robust_scaler(
    x,
):
    med = np.median(
        x,
        axis=0,
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75 -
        q25
    )

    iqr = np.where(
        iqr <
        1e-5,
        1.0,
        iqr,
    )

    return (
        med.astype(
            np.float32
        ),
        iqr.astype(
            np.float32
        ),
    )


def apply_robust_scaler(
    x,
    med,
    iqr,
):
    z = (
        x -
        med
    ) / iqr

    z = np.clip(
        z,
        -8.0,
        8.0,
    )

    return z.astype(
        np.float32
    )


CONTEXT_SCALED = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        train_mem = TASK_INDICES[
            key
        ][
            "train_memory"
        ]

        med, iqr = fit_robust_scaler(
            WINDOWS[
                key
            ][
                "context"
            ][
                train_mem
            ]
        )

        CONTEXT_SCALED[
            key
        ] = apply_robust_scaler(
            WINDOWS[
                key
            ][
                "context"
            ],
            med,
            iqr,
        )


## 14. GPU cosine Top-M within the same channel

In [ ]:

@torch.no_grad()
def cosine_topm(
    candidate_pattern,
    query_pattern,
    top_m,
):
    cand = torch.tensor(
        candidate_pattern,
        dtype=torch.float32,
        device=DEVICE,
    )

    query = torch.tensor(
        query_pattern,
        dtype=torch.float32,
        device=DEVICE,
    )

    idx_chunks = []
    score_chunks = []

    for start in range(
        0,
        len(
            query_pattern
        ),
        SEARCH_QUERY_BATCH,
    ):

        end = min(
            start +
            SEARCH_QUERY_BATCH,
            len(
                query_pattern
            ),
        )

        sim = (
            query[
                start:end
            ]
            @
            cand.T
        )

        score, idx = torch.topk(
            sim,
            k=top_m,
            dim=1,
            largest=True,
        )

        idx_chunks.append(
            idx.cpu()
        )

        score_chunks.append(
            score.cpu()
        )

        del sim

    return (
        torch.cat(
            idx_chunks,
            dim=0,
        ).numpy().astype(
            np.int64
        ),

        torch.cat(
            score_chunks,
            dim=0,
        ).numpy().astype(
            np.float32
        ),
    )


## 15. Build/load Same-channel Pattern Top-M pools

In [ ]:

PRESELECT = {}
pool_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        cache_path = (
            CACHE_DIR /
            f"{dataset_name}_H{H}_same_topM.npz"
        )

        if (
            cache_path.exists()
            and
            not FORCE_REBUILD_TOPM
        ):

            z = np.load(
                cache_path
            )

            out = {
                k:
                    z[
                        k
                    ]
                for k in z.files
            }

            source = "loaded"

        else:

            w = WINDOWS[
                key
            ]

            meta = w[
                "meta"
            ]

            out = {}

            for phase in [
                "train",
                "val",
                "test",
            ]:

                mem_idx = TASK_INDICES[
                    key
                ][
                    f"{phase}_memory"
                ]

                q_idx = TASK_INDICES[
                    key
                ][
                    f"{phase}_query"
                ]

                mem_channel = meta.iloc[
                    mem_idx
                ][
                    "ChannelIndex"
                ].to_numpy(
                    dtype=np.int64
                )

                q_channel = meta.iloc[
                    q_idx
                ][
                    "ChannelIndex"
                ].to_numpy(
                    dtype=np.int64
                )

                result_idx = np.empty(
                    (
                        len(
                            q_idx
                        ),
                        TOP_M,
                    ),
                    dtype=np.int64,
                )

                result_score = np.empty(
                    (
                        len(
                            q_idx
                        ),
                        TOP_M,
                    ),
                    dtype=np.float32,
                )

                for c in np.unique(
                    q_channel
                ):

                    q_pos = np.where(
                        q_channel ==
                        c
                    )[0]

                    mem_pos = np.where(
                        mem_channel ==
                        c
                    )[0]

                    assert len(
                        mem_pos
                    ) >= TOP_M

                    local_idx, score = cosine_topm(
                        w[
                            "pattern"
                        ][
                            mem_idx[
                                mem_pos
                            ]
                        ],
                        w[
                            "pattern"
                        ][
                            q_idx[
                                q_pos
                            ]
                        ],
                        TOP_M,
                    )

                    result_idx[
                        q_pos
                    ] = mem_idx[
                        mem_pos[
                            local_idx
                        ]
                    ]

                    result_score[
                        q_pos
                    ] = score

                out[
                    f"{phase}_query"
                ] = q_idx

                out[
                    f"{phase}_idx"
                ] = result_idx

                out[
                    f"{phase}_score"
                ] = result_score

            np.savez_compressed(
                cache_path,
                **out,
            )

            source = "built"

        PRESELECT[
            key
        ] = out

        pool_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "Source":
                source,

            "TrainQueries":
                len(
                    out[
                        "train_query"
                    ]
                ),

            "ValQueries":
                len(
                    out[
                        "val_query"
                    ]
                ),

            "TestQueries":
                len(
                    out[
                        "test_query"
                    ]
                ),
        })

pool_table = pd.DataFrame(
    pool_rows
)

display(
    pool_table
)

pool_table.to_csv(
    RESULT_DIR /
    "05_candidate_pool_summary.csv",
    index=False,
)


## 16. Candidate Prior reference statistics

In [ ]:

def build_prior_stats(
    dataset_name,
    H,
    reference_indices,
):
    key = (
        dataset_name,
        H
    )

    meta = WINDOWS[
        key
    ][
        "meta"
    ]

    future = WINDOWS[
        key
    ][
        "future"
    ]

    ref = np.asarray(
        reference_indices,
        dtype=np.int64,
    )

    ref_channel = meta.iloc[
        ref
    ][
        "ChannelIndex"
    ].to_numpy(
        dtype=np.int64
    )

    stats = {}

    for c in np.unique(
        ref_channel
    ):

        idx = ref[
            ref_channel ==
            c
        ]

        y = future[
            idx
        ]

        stats[
            int(
                c
            )
        ] = {
            "mean":
                y.mean(
                    axis=0
                ).astype(
                    np.float32
                ),

            "mean_sq":
                float(
                    np.mean(
                        y ** 2
                    )
                ),

            "count":
                int(
                    len(
                        y
                    )
                ),
        }

    return stats


PRIOR_STATS_TRAIN = {}
PRIOR_STATS_TRAINVAL = {}
prior_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        # Use all eligible historical query windows,
        # not only the model-training subsample.
        train_ref = FULL_PHASES[
            key
        ][
            "train_query"
        ]

        val_ref = FULL_PHASES[
            key
        ][
            "val_query"
        ]

        train_stats = build_prior_stats(
            dataset_name,
            H,
            train_ref,
        )

        trainval_stats = build_prior_stats(
            dataset_name,
            H,
            np.concatenate(
                [
                    train_ref,
                    val_ref,
                ]
            ),
        )

        PRIOR_STATS_TRAIN[
            key
        ] = train_stats

        PRIOR_STATS_TRAINVAL[
            key
        ] = trainval_stats

        for label, stats in [
            (
                "Train",
                train_stats,
            ),
            (
                "Train+Val",
                trainval_stats,
            ),
        ]:

            counts = [
                s[
                    "count"
                ]
                for s in stats.values()
            ]

            prior_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "ReferenceSet":
                    label,

                "Channels":
                    len(
                        stats
                    ),

                "MinReferenceWindowsPerChannel":
                    min(
                        counts
                    ),

                "MedianReferenceWindowsPerChannel":
                    float(
                        np.median(
                            counts
                        )
                    ),
            })

prior_reference_table = pd.DataFrame(
    prior_rows
)

display(
    prior_reference_table
)

prior_reference_table.to_csv(
    RESULT_DIR /
    "06_candidate_prior_reference.csv",
    index=False,
)


## 17. Candidate Prior matrix

In [ ]:

def candidate_prior_matrix(
    dataset_name,
    H,
    cand_idx,
    q_channel,
    prior_stats,
):
    key = (
        dataset_name,
        H
    )

    cand_future = WINDOWS[
        key
    ][
        "future"
    ][
        cand_idx
    ]

    B, M, HH = (
        cand_future.shape
    )

    out = np.empty(
        (
            B,
            M,
        ),
        dtype=np.float32,
    )

    for c in np.unique(
        q_channel
    ):

        pos = np.where(
            q_channel ==
            c
        )[0]

        assert int(
            c
        ) in prior_stats

        s = prior_stats[
            int(
                c
            )
        ]

        mu = s[
            "mean"
        ]

        mean_sq = s[
            "mean_sq"
        ]

        yi = cand_future[
            pos
        ]

        yi_sq = np.mean(
            yi ** 2,
            axis=2,
        )

        cross = np.mean(
            yi *
            mu[
                None,
                None,
                :
            ],
            axis=2,
        )

        out[
            pos
        ] = (
            mean_sq
            +
            yi_sq
            -
            2.0 *
            cross
        ).astype(
            np.float32
        )

    assert np.isfinite(
        out
    ).all()

    return out


## 18. Phase packaging

In [ ]:

def phase_data(
    dataset_name,
    H,
    phase,
    prior_stats,
):
    key = (
        dataset_name,
        H
    )

    w = WINDOWS[
        key
    ]

    p = PRESELECT[
        key
    ]

    q_idx = p[
        f"{phase}_query"
    ]

    cand_idx = p[
        f"{phase}_idx"
    ]

    meta = w[
        "meta"
    ]

    q_channel = meta.iloc[
        q_idx
    ][
        "ChannelIndex"
    ].to_numpy(
        dtype=np.int64
    )

    cand_channel = meta.iloc[
        cand_idx.reshape(
            -1
        )
    ][
        "ChannelIndex"
    ].to_numpy(
        dtype=np.int64
    ).reshape(
        cand_idx.shape
    )

    assert np.all(
        cand_channel
        ==
        q_channel[
            :,
            None
        ]
    )

    return {
        "q_idx":
            q_idx,

        "cand_idx":
            cand_idx,

        "pattern_score":
            p[
                f"{phase}_score"
            ].astype(
                np.float32
            ),

        "q_context":
            CONTEXT_SCALED[
                key
            ][
                q_idx
            ],

        "cand_context":
            CONTEXT_SCALED[
                key
            ][
                cand_idx
            ],

        "q_future":
            w[
                "future"
            ][
                q_idx
            ],

        "cand_future":
            w[
                "future"
            ][
                cand_idx
            ],

        "candidate_prior":
            candidate_prior_matrix(
                dataset_name,
                H,
                cand_idx,
                q_channel,
                prior_stats,
            ),

        "q_anchor":
            meta.iloc[
                q_idx
            ][
                "Anchor"
            ].to_numpy(
                dtype=np.int64
            ),

        "q_channel":
            q_channel,
    }


## 19. Retrieval metrics

In [ ]:

def future_distance(
    q_future,
    cand_future,
):
    return np.mean(
        (
            cand_future
            -
            q_future[
                :,
                None,
                :
            ]
        ) ** 2,
        axis=2,
    )


def topk_from_scores(
    score,
    k,
):
    idx = np.argpartition(
        -score,
        kth=k -
        1,
        axis=1,
    )[
        :,
        :k
    ]

    row = np.arange(
        len(
            score
        )
    )[
        :,
        None
    ]

    order = np.argsort(
        -score[
            row,
            idx
        ],
        axis=1,
    )

    return idx[
        row,
        order
    ]


def gather2(
    x,
    idx,
):
    row = np.arange(
        len(
            x
        )
    )[
        :,
        None
    ]

    return x[
        row,
        idx
    ]


def gather3(
    x,
    idx,
):
    row = np.arange(
        len(
            x
        )
    )[
        :,
        None
    ]

    return x[
        row,
        idx,
        :
    ]


def ndcg_at_k(
    score,
    fdist,
    k,
):
    mean = fdist.mean(
        axis=1,
        keepdims=True,
    )

    std = (
        fdist.std(
            axis=1,
            keepdims=True,
        )
        +
        1e-6
    )

    z = (
        fdist -
        mean
    ) / std

    relevance = np.exp(
        -z /
        TAU_Y
    )

    selected = topk_from_scores(
        score,
        k,
    )

    ideal = np.argsort(
        -relevance,
        axis=1,
    )[
        :,
        :k
    ]

    discount = (
        1.0
        /
        np.log2(
            np.arange(
                2,
                k +
                2
            )
        )
    )[
        None,
        :
    ]

    dcg = np.sum(
        gather2(
            relevance,
            selected,
        )
        *
        discount,
        axis=1,
    )

    idcg = (
        np.sum(
            gather2(
                relevance,
                ideal,
            )
            *
            discount,
            axis=1,
        )
        +
        EPS
    )

    return (
        dcg /
        idcg
    ).astype(
        np.float32
    )


def oracle_recall_at_k(
    selected,
    fdist,
    k,
):
    oracle = np.argpartition(
        fdist,
        kth=k -
        1,
        axis=1,
    )[
        :,
        :k
    ]

    out = np.empty(
        len(
            selected
        ),
        dtype=np.float32,
    )

    for i in range(
        len(
            selected
        )
    ):

        out[
            i
        ] = (
            len(
                set(
                    selected[
                        i
                    ].tolist()
                )
                &
                set(
                    oracle[
                        i
                    ].tolist()
                )
            )
            /
            k
        )

    return out


def query_metrics(
    score,
    phase,
):
    fdist = future_distance(
        phase[
            "q_future"
        ],
        phase[
            "cand_future"
        ],
    )

    selected = topk_from_scores(
        score,
        TOP_K,
    )

    analog = gather2(
        fdist,
        selected,
    ).mean(
        axis=1
    )

    selected_future = gather3(
        phase[
            "cand_future"
        ],
        selected,
    )

    pred = selected_future.mean(
        axis=1
    )

    forecast = np.mean(
        (
            pred
            -
            phase[
                "q_future"
            ]
        ) ** 2,
        axis=1,
    )

    return pd.DataFrame({
        "AnalogFutureMSE":
            analog.astype(
                np.float32
            ),

        "RetrievalForecastMSE":
            forecast.astype(
                np.float32
            ),

        "NDCG@K":
            ndcg_at_k(
                score,
                fdist,
                TOP_K,
            ),

        "OracleRecall@K":
            oracle_recall_at_k(
                selected,
                fdist,
                TOP_K,
            ),
    })


## 20. Future-Compatible retriever

In [ ]:

class FutureCompatibleReranker(
    nn.Module
):
    def __init__(
        self,
        context_dim=7,
        hidden_dim=128,
        dropout=0.10,
        initial_alpha=0.10,
    ):
        super().__init__()

        input_dim = (
            1
            +
            4 *
            context_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                hidden_dim //
                2,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim //
                2,
                1,
            ),
        )

        raw_alpha = math.log(
            math.exp(
                initial_alpha
            )
            -
            1.0
        )

        self.raw_alpha = nn.Parameter(
            torch.tensor(
                raw_alpha,
                dtype=torch.float32,
            )
        )

    @property
    def alpha(
        self
    ):
        return F.softplus(
            self.raw_alpha
        )

    def forward(
        self,
        pattern_score,
        q_context,
        cand_context,
    ):
        B, M, D = (
            cand_context.shape
        )

        q = (
            q_context[
                :,
                None,
                :
            ]
            .expand(
                -1,
                M,
                -1,
            )
        )

        diff = (
            q -
            cand_context
        )

        feat = torch.cat(
            [
                pattern_score[
                    ...,
                    None
                ],
                q,
                cand_context,
                diff,
                diff.abs(),
            ],
            dim=-1,
        )

        delta = (
            self.mlp(
                feat
            )
            .squeeze(
                -1
            )
        )

        return (
            pattern_score
            +
            self.alpha *
            delta
        )


## 21. Listwise future-compatibility loss

In [ ]:

def listwise_future_loss(
    score,
    future_dist,
):
    mean = future_dist.mean(
        dim=1,
        keepdim=True,
    )

    std = future_dist.std(
        dim=1,
        keepdim=True,
        unbiased=False,
    ).clamp_min(
        1e-6
    )

    z = (
        future_dist
        -
        mean
    ) / std

    target = torch.softmax(
        -z /
        TAU_Y,
        dim=1,
    )

    log_prob = F.log_softmax(
        score,
        dim=1,
    )

    loss = -(
        target *
        log_prob
    ).sum(
        dim=1
    ).mean()

    assert torch.isfinite(
        loss
    )

    return loss


## 22. Strong within-channel shuffled-future control

In [ ]:

def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def within_channel_cyclic_shuffle(
    phase,
    seed,
):
    """
    Marginal-preserving negative control.

    Each channel is circularly shifted by a random non-zero amount.
    This guarantees no query keeps its original future when
    the channel contains at least two queries.
    """
    rng = np.random.default_rng(
        seed
    )

    original = phase[
        "q_future"
    ]

    out = original.copy()

    channels = phase[
        "q_channel"
    ]

    for c in np.unique(
        channels
    ):

        pos = np.where(
            channels ==
            c
        )[0]

        if len(
            pos
        ) <= 1:
            continue

        shift = int(
            rng.integers(
                1,
                len(
                    pos
                ),
            )
        )

        src = np.roll(
            pos,
            shift,
        )

        out[
            pos
        ] = original[
            src
        ]

    return out


## 23. Training and scoring helpers

In [ ]:

def train_one_phase(
    model,
    optimizer,
    phase,
    shuffled_future=None,
):
    model.train()

    n = len(
        phase[
            "q_idx"
        ]
    )

    order = np.random.permutation(
        n
    )

    losses = []

    for start in range(
        0,
        n,
        TRAIN_BATCH,
    ):

        ids = order[
            start:
            start +
            TRAIN_BATCH
        ]

        ps = torch.tensor(
            phase[
                "pattern_score"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qc = torch.tensor(
            phase[
                "q_context"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cc = torch.tensor(
            phase[
                "cand_context"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cf = torch.tensor(
            phase[
                "cand_future"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qf_np = (
            phase[
                "q_future"
            ][
                ids
            ]
            if shuffled_future is None
            else
            shuffled_future[
                ids
            ]
        )

        qf = torch.tensor(
            qf_np,
            dtype=torch.float32,
            device=DEVICE,
        )

        dist = (
            (
                cf
                -
                qf[
                    :,
                    None,
                    :
                ]
            ) ** 2
        ).mean(
            dim=2
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        score = model(
            ps,
            qc,
            cc,
        )

        loss = listwise_future_loss(
            score,
            dist,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0,
        )

        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def predict_scores(
    model,
    phase,
):
    model.eval()

    chunks = []

    n = len(
        phase[
            "q_idx"
        ]
    )

    for start in range(
        0,
        n,
        EVAL_BATCH,
    ):

        end = min(
            start +
            EVAL_BATCH,
            n,
        )

        ps = torch.tensor(
            phase[
                "pattern_score"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qc = torch.tensor(
            phase[
                "q_context"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cc = torch.tensor(
            phase[
                "cand_context"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        chunks.append(
            model(
                ps,
                qc,
                cc,
            ).cpu()
        )

    return (
        torch.cat(
            chunks,
            dim=0,
        )
        .numpy()
        .astype(
            np.float32
        )
    )


## 24. Phase-A epoch selection

In [ ]:

def select_epoch(
    dataset_name,
    H,
    seed,
    shuffled=False,
):
    key = (
        dataset_name,
        H
    )

    set_seed(
        seed
    )

    train_phase = phase_data(
        dataset_name,
        H,
        "train",
        PRIOR_STATS_TRAIN[
            key
        ],
    )

    val_phase = phase_data(
        dataset_name,
        H,
        "val",
        PRIOR_STATS_TRAIN[
            key
        ],
    )

    model = FutureCompatibleReranker(
        context_dim=CONTEXT_DIM
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    shuffled_train = None

    if shuffled:

        shuffled_train = within_channel_cyclic_shuffle(
            train_phase,
            seed=(
                100000
                +
                seed
            ),
        )

    best_epoch = None
    best_val = float(
        "inf"
    )

    wait = 0
    rows = []

    for epoch in range(
        1,
        MAX_EPOCHS +
        1,
    ):

        train_loss = train_one_phase(
            model,
            optimizer,
            train_phase,
            shuffled_future=shuffled_train,
        )

        val_score = predict_scores(
            model,
            val_phase,
        )

        val_m = query_metrics(
            val_score,
            val_phase,
        )

        val_analog = float(
            val_m[
                "AnalogFutureMSE"
            ].mean()
        )

        rows.append({
            "Epoch":
                epoch,

            "TrainLoss":
                train_loss,

            "ValAnalogFutureMSE":
                val_analog,

            "ValNDCG@K":
                float(
                    val_m[
                        "NDCG@K"
                    ].mean()
                ),

            "Alpha":
                float(
                    model.alpha.item()
                ),
        })

        if (
            val_analog
            <
            best_val
            -
            1e-10
        ):

            best_val = val_analog
            best_epoch = epoch
            wait = 0

        else:

            wait += 1

        if wait >= PATIENCE:
            break

    assert best_epoch is not None

    return (
        best_epoch,
        best_val,
        pd.DataFrame(
            rows
        ),
    )


## 25. Phase-B refit on Train + Validation retrieval supervision

In [ ]:

def refit_model(
    dataset_name,
    H,
    seed,
    epochs,
    shuffled=False,
):
    key = (
        dataset_name,
        H
    )

    set_seed(
        seed
    )

    train_phase = phase_data(
        dataset_name,
        H,
        "train",
        PRIOR_STATS_TRAINVAL[
            key
        ],
    )

    val_phase = phase_data(
        dataset_name,
        H,
        "val",
        PRIOR_STATS_TRAINVAL[
            key
        ],
    )

    model = FutureCompatibleReranker(
        context_dim=CONTEXT_DIM
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    shuffled_train = None
    shuffled_val = None

    if shuffled:

        shuffled_train = within_channel_cyclic_shuffle(
            train_phase,
            seed=(
                200000
                +
                seed
            ),
        )

        shuffled_val = within_channel_cyclic_shuffle(
            val_phase,
            seed=(
                300000
                +
                seed
            ),
        )

    for _ in range(
        epochs
    ):

        train_one_phase(
            model,
            optimizer,
            train_phase,
            shuffled_future=shuffled_train,
        )

        train_one_phase(
            model,
            optimizer,
            val_phase,
            shuffled_future=shuffled_val,
        )

    return model


## 26. Train/load Learned and Shuffled models

In [ ]:

MODELS = {}
training_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        for method_name, shuffled in [
            (
                "Learned",
                False,
            ),
            (
                "ShuffledFuture",
                True,
            ),
        ]:

            for seed in SEEDS:

                path = (
                    MODEL_DIR /
                    f"{dataset_name}_H{H}_{method_name}_seed{seed}.pt"
                )

                history_path = (
                    MODEL_DIR /
                    f"{dataset_name}_H{H}_{method_name}_seed{seed}_history.csv"
                )

                if (
                    path.exists()
                    and
                    not FORCE_RETRAIN
                ):

                    ckpt = torch.load(
                        path,
                        map_location="cpu",
                        weights_only=False,
                    )

                    model = FutureCompatibleReranker(
                        context_dim=CONTEXT_DIM
                    ).to(
                        DEVICE
                    )

                    model.load_state_dict(
                        ckpt[
                            "StateDict"
                        ]
                    )

                    best_epoch = int(
                        ckpt[
                            "BestEpoch"
                        ]
                    )

                    best_val = float(
                        ckpt[
                            "BestValidationAnalogFutureMSE"
                        ]
                    )

                    source = "loaded"

                else:

                    (
                        best_epoch,
                        best_val,
                        history,
                    ) = select_epoch(
                        dataset_name,
                        H,
                        seed,
                        shuffled=shuffled,
                    )

                    history[
                        "Dataset"
                    ] = dataset_name

                    history[
                        "Horizon"
                    ] = H

                    history[
                        "Method"
                    ] = method_name

                    history[
                        "Seed"
                    ] = seed

                    history.to_csv(
                        history_path,
                        index=False,
                    )

                    model = refit_model(
                        dataset_name,
                        H,
                        seed,
                        best_epoch,
                        shuffled=shuffled,
                    )

                    torch.save(
                        {
                            "Dataset":
                                dataset_name,

                            "Horizon":
                                H,

                            "Method":
                                method_name,

                            "Seed":
                                seed,

                            "BestEpoch":
                                best_epoch,

                            "BestValidationAnalogFutureMSE":
                                best_val,

                            "StateDict":
                                model.state_dict(),

                            "Protocol":
                                (
                                    "Same-channel; fixed train-scale; "
                                    "M100 K10; Phase-A val selection; "
                                    "Phase-B train+val refit"
                                ),
                        },
                        path,
                    )

                    source = "trained"

                model.eval()

                MODELS[
                    (
                        dataset_name,
                        H,
                        method_name,
                        seed,
                    )
                ] = model

                training_rows.append({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Method":
                        method_name,

                    "Seed":
                        seed,

                    "BestEpoch":
                        best_epoch,

                    "BestValidationAnalogFutureMSE":
                        best_val,

                    "Alpha":
                        float(
                            model.alpha.item()
                        ),

                    "Source":
                        source,
                })

                print(
                    dataset_name,
                    "H=",
                    H,
                    method_name,
                    "seed=",
                    seed,
                    "|",
                    source,
                    "| epoch",
                    best_epoch,
                    "| val",
                    best_val,
                )

training_table = pd.DataFrame(
    training_rows
)

training_table.to_csv(
    RESULT_DIR /
    "07_training_summary.csv",
    index=False,
)


## 27. Final Test evaluation

In [ ]:

QUERY_RESULTS = {}
seed_rows = []
main_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        test_phase = phase_data(
            dataset_name,
            H,
            "test",
            PRIOR_STATS_TRAINVAL[
                key
            ],
        )

        # Pattern
        pattern_m = query_metrics(
            test_phase[
                "pattern_score"
            ],
            test_phase,
        )

        # Candidate-global utility
        prior_score = -test_phase[
            "candidate_prior"
        ]

        prior_m = query_metrics(
            prior_score,
            test_phase,
        )

        seed_metrics = {}

        for method_name in [
            "Learned",
            "ShuffledFuture",
        ]:

            frames = []

            for seed in SEEDS:

                model = MODELS[
                    (
                        dataset_name,
                        H,
                        method_name,
                        seed,
                    )
                ]

                score = predict_scores(
                    model,
                    test_phase,
                )

                m = query_metrics(
                    score,
                    test_phase,
                )

                frames.append(
                    m
                )

                seed_rows.append({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Method":
                        method_name,

                    "Seed":
                        seed,

                    "AnalogFutureMSE":
                        float(
                            m[
                                "AnalogFutureMSE"
                            ].mean()
                        ),

                    "RetrievalForecastMSE":
                        float(
                            m[
                                "RetrievalForecastMSE"
                            ].mean()
                        ),

                    "NDCG@K":
                        float(
                            m[
                                "NDCG@K"
                            ].mean()
                        ),

                    "OracleRecall@K":
                        float(
                            m[
                                "OracleRecall@K"
                            ].mean()
                        ),
                })

            seed_metrics[
                method_name
            ] = pd.DataFrame({
                col:
                    np.stack(
                        [
                            f[
                                col
                            ].to_numpy()
                            for f in frames
                        ],
                        axis=0,
                    ).mean(
                        axis=0
                    )

                for col in frames[
                    0
                ].columns
            })

        q = pd.DataFrame({
            "Anchor":
                test_phase[
                    "q_anchor"
                ],

            "ChannelIndex":
                test_phase[
                    "q_channel"
                ],
        })

        method_metrics = {
            "Pattern":
                pattern_m,

            "CandidatePrior":
                prior_m,

            "Learned":
                seed_metrics[
                    "Learned"
                ],

            "ShuffledFuture":
                seed_metrics[
                    "ShuffledFuture"
                ],
        }

        for method_name, m in method_metrics.items():

            for metric in m.columns:

                q[
                    f"{method_name}_{metric}"
                ] = m[
                    metric
                ].to_numpy()

            main_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Method":
                    method_name,

                "AnalogFutureMSE":
                    float(
                        m[
                            "AnalogFutureMSE"
                        ].mean()
                    ),

                "RetrievalForecastMSE":
                    float(
                        m[
                            "RetrievalForecastMSE"
                        ].mean()
                    ),

                "NDCG@K":
                    float(
                        m[
                            "NDCG@K"
                        ].mean()
                    ),

                "OracleRecall@K":
                    float(
                        m[
                            "OracleRecall@K"
                        ].mean()
                    ),
            })

        QUERY_RESULTS[
            key
        ] = q

        q.to_csv(
            RESULT_DIR /
            f"query_level_{dataset_name}_H{H}.csv.gz",
            index=False,
            compression="gzip",
        )

seed_table = pd.DataFrame(
    seed_rows
)

main_table = pd.DataFrame(
    main_rows
)

display(
    main_table.sort_values(
        [
            "Dataset",
            "Horizon",
            "AnalogFutureMSE",
        ]
    )
)

seed_table.to_csv(
    RESULT_DIR /
    "08_seed_results.csv",
    index=False,
)

main_table.to_csv(
    RESULT_DIR /
    "09_main_confirmatory_summary.csv",
    index=False,
)


## 28. Relative evidence table

In [ ]:

relative_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        t = (
            main_table[
                (
                    main_table[
                        "Dataset"
                    ] ==
                    dataset_name
                )
                &
                (
                    main_table[
                        "Horizon"
                    ] ==
                    H
                )
            ]
            .set_index(
                "Method"
            )
        )

        def val(
            method,
            metric="AnalogFutureMSE",
        ):
            return float(
                t.loc[
                    method,
                    metric,
                ]
            )

        pattern = val(
            "Pattern"
        )

        prior = val(
            "CandidatePrior"
        )

        learned = val(
            "Learned"
        )

        shuffled = val(
            "ShuffledFuture"
        )

        relative_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "Pattern":
                pattern,

            "CandidatePrior":
                prior,

            "Learned":
                learned,

            "ShuffledFuture":
                shuffled,

            "Pattern_to_Learned_%":
                100.0 *
                (
                    pattern -
                    learned
                ) /
                pattern,

            "Pattern_to_CandidatePrior_%":
                100.0 *
                (
                    pattern -
                    prior
                ) /
                pattern,

            "Shuffled_to_Learned_%":
                100.0 *
                (
                    shuffled -
                    learned
                ) /
                shuffled,

            "CandidatePrior_to_Learned_%":
                100.0 *
                (
                    prior -
                    learned
                ) /
                prior,

            "Learned_NDCG@K":
                val(
                    "Learned",
                    "NDCG@K",
                ),

            "Shuffled_NDCG@K":
                val(
                    "ShuffledFuture",
                    "NDCG@K",
                ),

            "Pattern_NDCG@K":
                val(
                    "Pattern",
                    "NDCG@K",
                ),

            "CandidatePrior_NDCG@K":
                val(
                    "CandidatePrior",
                    "NDCG@K",
                ),
        })

relative_table = pd.DataFrame(
    relative_rows
)

display(
    relative_table
)

relative_table.to_csv(
    RESULT_DIR /
    "10_relative_evidence.csv",
    index=False,
)


## 29. Seed stability

In [ ]:

stability_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        for method_name in [
            "Learned",
            "ShuffledFuture",
        ]:

            s = seed_table[
                (
                    seed_table[
                        "Dataset"
                    ] ==
                    dataset_name
                )
                &
                (
                    seed_table[
                        "Horizon"
                    ] ==
                    H
                )
                &
                (
                    seed_table[
                        "Method"
                    ] ==
                    method_name
                )
            ][
                "AnalogFutureMSE"
            ]

            mean = float(
                s.mean()
            )

            std = float(
                s.std(
                    ddof=1
                )
            )

            stability_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Method":
                    method_name,

                "Mean":
                    mean,

                "Std":
                    std,

                "CV_%":
                    100.0 *
                    std /
                    (
                        mean +
                        EPS
                    ),

                "Min":
                    float(
                        s.min()
                    ),

                "Max":
                    float(
                        s.max()
                    ),
            })

stability_table = pd.DataFrame(
    stability_rows
)

display(
    stability_table
)

stability_table.to_csv(
    RESULT_DIR /
    "11_seed_stability.csv",
    index=False,
)


## 30. Moving-block bootstrap

In [ ]:

def moving_block_bootstrap(
    x,
    block_len,
    n_boot,
    seed,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    n = len(
        x
    )

    assert n >= block_len

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n /
        block_len
    )

    max_start = (
        n -
        block_len
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):

        parts = []

        for _ in range(
            n_blocks
        ):

            start = rng.integers(
                0,
                max_start +
                1,
            )

            parts.append(
                x[
                    start:
                    start +
                    block_len
                ]
            )

        sample = np.concatenate(
            parts
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "ObservedImprovement":
            float(
                x.mean()
            ),

        "CI_2.5%":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),

        "CI_97.5%":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),

        "P_gt_0":
            float(
                (
                    boot >
                    0
                ).mean()
            ),
    }


In [ ]:

bootstrap_rows = []

comparisons = [
    (
        "Pattern",
        "Learned",
    ),
    (
        "ShuffledFuture",
        "Learned",
    ),
    (
        "CandidatePrior",
        "Learned",
    ),
    (
        "Pattern",
        "CandidatePrior",
    ),
]

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        q = QUERY_RESULTS[
            (
                dataset_name,
                H
            )
        ]

        for baseline, proposed in comparisons:

            for metric in [
                "AnalogFutureMSE",
                "RetrievalForecastMSE",
            ]:

                tmp = pd.DataFrame({
                    "Anchor":
                        q[
                            "Anchor"
                        ],

                    "Diff":
                        (
                            q[
                                f"{baseline}_{metric}"
                            ]
                            -
                            q[
                                f"{proposed}_{metric}"
                            ]
                        ),
                })

                anchor_diff = (
                    tmp
                    .groupby(
                        "Anchor"
                    )[
                        "Diff"
                    ]
                    .mean()
                    .sort_index()
                    .to_numpy()
                )

                r = moving_block_bootstrap(
                    anchor_diff,
                    BLOCK_ANCHORS,
                    N_BOOT,
                    seed=(
                        DATASET_SEED[
                            dataset_name
                        ]
                        +
                        H *
                        100
                        +
                        len(
                            bootstrap_rows
                        )
                    ),
                )

                r.update({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Baseline":
                        baseline,

                    "Proposed":
                        proposed,

                    "Metric":
                        metric,

                    "SignificantImprovement":
                        bool(
                            r[
                                "CI_2.5%"
                            ] >
                            0
                        ),
                })

                bootstrap_rows.append(
                    r
                )

bootstrap_table = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_table
)

bootstrap_table.to_csv(
    RESULT_DIR /
    "12_moving_block_bootstrap.csv",
    index=False,
)


## 31. Task-level relevance taxonomy

In [ ]:
def bootstrap_sig(
    dataset_name,
    H,
    baseline,
    proposed,
    metric="AnalogFutureMSE",
):
    x = bootstrap_table[
        (
            bootstrap_table[
                "Dataset"
            ] ==
            dataset_name
        )
        &
        (
            bootstrap_table[
                "Horizon"
            ] ==
            H
        )
        &
        (
            bootstrap_table[
                "Baseline"
            ] ==
            baseline
        )
        &
        (
            bootstrap_table[
                "Proposed"
            ] ==
            proposed
        )
        &
        (
            bootstrap_table[
                "Metric"
            ] ==
            metric
        )
    ]

    assert len(x) == 1

    return bool(
        x[
            "SignificantImprovement"
        ].iloc[0]
    )


taxonomy_rows = []

# Column names containing "%" are not preserved as attributes by itertuples().
# iterrows() keeps the exact column names.
for _, row in relative_table.iterrows():

    dataset_name = row[
        "Dataset"
    ]

    H = int(
        row[
            "Horizon"
        ]
    )

    pattern_to_learned = float(
        row[
            "Pattern_to_Learned_%"
        ]
    )

    pattern_to_prior = float(
        row[
            "Pattern_to_CandidatePrior_%"
        ]
    )

    shuffled_to_learned = float(
        row[
            "Shuffled_to_Learned_%"
        ]
    )

    prior_to_learned = float(
        row[
            "CandidatePrior_to_Learned_%"
        ]
    )

    query_specific = (
        shuffled_to_learned >
        0
    )

    global_utility = (
        pattern_to_prior >
        0
    )

    learned_beyond_prior = (
        prior_to_learned >
        0
    )

    sig_query = bootstrap_sig(
        dataset_name,
        H,
        "ShuffledFuture",
        "Learned",
    )

    sig_global = bootstrap_sig(
        dataset_name,
        H,
        "Pattern",
        "CandidatePrior",
    )

    sig_pattern_learned = bootstrap_sig(
        dataset_name,
        H,
        "Pattern",
        "Learned",
    )

    if (
        global_utility
        and
        sig_global
        and
        (
            not learned_beyond_prior
        )
    ):

        regime = (
            "Candidate-global-dominant"
        )

    elif (
        query_specific
        and
        sig_query
        and
        (
            not global_utility
            or
            learned_beyond_prior
        )
    ):

        regime = (
            "Query-specific-dominant"
        )

    elif (
        global_utility
        and
        query_specific
    ):

        regime = "Mixed"

    elif (
        sig_pattern_learned
    ):

        regime = (
            "Learned relevance, mechanism mixed"
        )

    else:

        regime = (
            "No strong extra relevance evidence"
        )

    taxonomy_rows.append({
        "Dataset":
            dataset_name,

        "Horizon":
            H,

        "PatternToLearned_%":
            pattern_to_learned,

        "PatternToPrior_%":
            pattern_to_prior,

        "ShuffledToLearned_%":
            shuffled_to_learned,

        "PriorToLearned_%":
            prior_to_learned,

        "Sig_PatternToLearned":
            sig_pattern_learned,

        "Sig_ShuffledToLearned":
            sig_query,

        "Sig_PatternToPrior":
            sig_global,

        "Regime":
            regime,
    })

taxonomy_table = pd.DataFrame(
    taxonomy_rows
)

display(
    taxonomy_table
)

taxonomy_table.to_csv(
    RESULT_DIR /
    "13_task_relevance_taxonomy.csv",
    index=False,
)


## 32. Dataset-level evidence summary

In [ ]:

dataset_rows = []

for dataset_name in DATASET_NAMES:

    x = taxonomy_table[
        taxonomy_table[
            "Dataset"
        ] ==
        dataset_name
    ]

    dataset_rows.append({
        "Dataset":
            dataset_name,

        "Horizons":
            len(
                x
            ),

        "LearnedBeatsPattern":
            int(
                (
                    x[
                        "PatternToLearned_%"
                    ] >
                    0
                ).sum()
            ),

        "SignificantLearnedVsPattern":
            int(
                x[
                    "Sig_PatternToLearned"
                ].sum()
            ),

        "LearnedBeatsShuffled":
            int(
                (
                    x[
                        "ShuffledToLearned_%"
                    ] >
                    0
                ).sum()
            ),

        "SignificantLearnedVsShuffled":
            int(
                x[
                    "Sig_ShuffledToLearned"
                ].sum()
            ),

        "PriorBeatsPattern":
            int(
                (
                    x[
                        "PatternToPrior_%"
                    ] >
                    0
                ).sum()
            ),

        "SignificantPriorVsPattern":
            int(
                x[
                    "Sig_PatternToPrior"
                ].sum()
            ),

        "LearnedBeatsPrior":
            int(
                (
                    x[
                        "PriorToLearned_%"
                    ] >
                    0
                ).sum()
            ),

        "MeanPatternToLearned_%":
            float(
                x[
                    "PatternToLearned_%"
                ].mean()
            ),

        "MeanShuffledToLearned_%":
            float(
                x[
                    "ShuffledToLearned_%"
                ].mean()
            ),

        "MeanPatternToPrior_%":
            float(
                x[
                    "PatternToPrior_%"
                ].mean()
            ),
    })

dataset_summary = pd.DataFrame(
    dataset_rows
)

display(
    dataset_summary
)

dataset_summary.to_csv(
    RESULT_DIR /
    "14_dataset_evidence_summary.csv",
    index=False,
)


## 33. New-dataset evidence check

In [ ]:

new_dataset_rows = []

for dataset_name in [
    "Exchange",
    "Solar",
]:

    x = taxonomy_table[
        taxonomy_table[
            "Dataset"
        ] ==
        dataset_name
    ]

    new_dataset_rows.append({
        "Dataset":
            dataset_name,

        "QuerySpecificEvidenceHorizons":
            int(
                x[
                    "Sig_ShuffledToLearned"
                ].sum()
            ),

        "CandidateGlobalEvidenceHorizons":
            int(
                x[
                    "Sig_PatternToPrior"
                ].sum()
            ),

        "LearnedBeyondPriorHorizons":
            int(
                (
                    x[
                        "PriorToLearned_%"
                    ] >
                    0
                ).sum()
            ),

        "Regimes":
            " | ".join(
                x[
                    "Regime"
                ].tolist()
            ),
    })

new_dataset_table = pd.DataFrame(
    new_dataset_rows
)

display(
    new_dataset_table
)

new_dataset_table.to_csv(
    RESULT_DIR /
    "15_new_dataset_evidence.csv",
    index=False,
)


## 34. Final confirmatory decision

In [ ]:

total_tasks = len(
    taxonomy_table
)

learned_pattern_wins = int(
    (
        taxonomy_table[
            "PatternToLearned_%"
        ] >
        0
    ).sum()
)

learned_pattern_sig = int(
    taxonomy_table[
        "Sig_PatternToLearned"
    ].sum()
)

query_specific_sig = int(
    taxonomy_table[
        "Sig_ShuffledToLearned"
    ].sum()
)

global_prior_sig = int(
    taxonomy_table[
        "Sig_PatternToPrior"
    ].sum()
)

exchange_has_mechanism = bool(
    (
        taxonomy_table[
            taxonomy_table[
                "Dataset"
            ] ==
            "Exchange"
        ][
            [
                "Sig_ShuffledToLearned",
                "Sig_PatternToPrior",
            ]
        ]
        .any(
            axis=1
        )
        .any()
    )
)

solar_has_mechanism = bool(
    (
        taxonomy_table[
            taxonomy_table[
                "Dataset"
            ] ==
            "Solar"
        ][
            [
                "Sig_ShuffledToLearned",
                "Sig_PatternToPrior",
            ]
        ]
        .any(
            axis=1
        )
        .any()
    )
)

electricity_traffic_query_sig = int(
    taxonomy_table[
        taxonomy_table[
            "Dataset"
        ].isin(
            [
                "Electricity",
                "Traffic",
            ]
        )
    ][
        "Sig_ShuffledToLearned"
    ].sum()
)

if (
    exchange_has_mechanism
    and
    solar_has_mechanism
    and
    (
        query_specific_sig > 0
        or
        global_prior_sig > 0
    )
):

    recommendation = (
        "Confirmatory benchmark evidence is sufficient. "
        "Stop benchmark model tuning and proceed to the final evidence map, "
        "theoretical formulation, figures/tables, and ICLR manuscript."
    )

else:

    recommendation = (
        "The confirmatory experiment is informative but one new dataset shows weak mechanism evidence. "
        "Do not tune the model to fix it. Preserve it as a negative/weak case and proceed with a qualified claim."
    )

decision = pd.DataFrame(
    [
        {
            "TotalTasks":
                total_tasks,

            "LearnedBeatsPattern":
                learned_pattern_wins,

            "SignificantLearnedVsPattern":
                learned_pattern_sig,

            "SignificantQuerySpecificEvidence":
                query_specific_sig,

            "SignificantCandidateGlobalEvidence":
                global_prior_sig,

            "ElectricityTraffic_SignificantQuerySpecific":
                electricity_traffic_query_sig,

            "ExchangeHasSignificantMechanismEvidence":
                exchange_has_mechanism,

            "SolarHasSignificantMechanismEvidence":
                solar_has_mechanism,

            "Recommendation":
                recommendation,
        }
    ]
)

display(
    decision
)

decision.to_csv(
    RESULT_DIR /
    "16_final_confirmatory_decision.csv",
    index=False,
)


# Result Reading Guide

Default output directory:

```text
_work/final_confirmatory/
```

Key result files:

```text
00_data_manifest.csv
09_main_confirmatory_summary.csv
10_relative_evidence.csv
11_seed_stability.csv
12_moving_block_bootstrap.csv
13_task_relevance_taxonomy.csv
14_dataset_evidence_summary.csv
15_new_dataset_evidence.csv
16_final_confirmatory_decision.csv
```

## Exchange

Inspect the Shuffled-to-Learned comparison first. A positive and statistically significant gain is evidence of query-specific future compatibility; a strong Candidate Prior instead indicates candidate-global or mixed relevance.

## Solar

Inspect both Pattern-to-Candidate-Prior and Shuffled-to-Learned. The former diagnoses candidate-global utility; the latter diagnoses query-specific relevance.

## Electricity / Traffic

The key confirmatory ordering is

\[
\boxed{\text{Learned}<\text{Within-channel Shuffled}}
\]

under the same-channel, fixed train-scale protocol.

## Dataset taxonomy

`13_task_relevance_taxonomy.csv` summarizes the controls using the following analysis labels:

```text
Candidate-global-dominant
Query-specific-dominant
Mixed
Learned relevance, mechanism mixed
No strong extra relevance evidence
```

These labels are analysis aids. Paper claims should be based on the numerical values and moving-block bootstrap intervals.

## Stopping rule

The benchmark recipe is not retuned after this confirmatory experiment. Negative or mixed results are informative because the paper studies domain-dependent relevance structure.
